## 1. Mount Google Drive (do this first — protects against session crashes)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/aimo3'
os.makedirs(DRIVE_DIR, exist_ok=True)

OUTPUT_FILE = f'{DRIVE_DIR}/nemotron_training_data.jsonl'
METADATA_FILE = f'{DRIVE_DIR}/nemotron_training_metadata.json'
CHECKPOINT_FILE = f'{DRIVE_DIR}/pipeline_checkpoint.json'

print(f'✓ Drive mounted. Output will be saved to: {DRIVE_DIR}')

## 2. Install and Import

In [ ]:
!pip install -q datasets huggingface_hub pandas numpy tqdm

In [ ]:
import json
import os
import re
import random
import numpy as np
import pandas as pd
from collections import defaultdict
from datasets import load_dataset
from tqdm.auto import tqdm

random.seed(42)
np.random.seed(42)
print('✓ Imports successful')

## 3. HuggingFace Login

In [ ]:
from huggingface_hub import login
from google.colab import userdata

# Store your HF token in Colab Secrets (left sidebar → key icon) as 'HF_TOKEN'
# This survives session restarts unlike interactive login
try:
    token = userdata.get('HF_TOKEN')
    login(token=token, add_to_git_credential=False)
    print('✓ Logged in via Colab secret')
except Exception:
    # Fallback to interactive login
    print('HF_TOKEN secret not found. Using interactive login.')
    login()

## 4. Configuration

In [ ]:
# Dataset config
DATASET_ID = 'nvidia/Nemotron-Math-v2'

# Target distribution
TARGET_DISTRIBUTION = {
    'easy':   {'total': 3000,  'tir_ratio': 0.30},
    'medium': {'total': 7000,  'tir_ratio': 0.60},
    'hard':   {'total': 10000, 'tir_ratio': 0.85},
}

# Adaptive sampling behavior
ALLOW_NON_INT_FALLBACK_ALL = True   # if True, easy/medium can also backfill from non-integer answers
FILL_GLOBAL_SHORTFALL = True        # if True, remaining target is filled from any leftover pools

# Difficulty thresholds
EASY_MIN      = 0.75
MEDIUM_MIN    = 0.30
HARD_TOOL_MIN = 0.25

# Answer range
ANSWER_MIN = 0
ANSWER_MAX = 99999

total = sum(v['total'] for v in TARGET_DISTRIBUTION.values())
print(f'Config loaded - target: {total:,} total samples')
for diff, p_cfg in TARGET_DISTRIBUTION.items():
    tir = int(p_cfg['total'] * p_cfg['tir_ratio'])
    print(f'  {diff:8}: {p_cfg["total"]:,} ({tir:,} TIR + {p_cfg["total"]-tir:,} no-TIR)')
print(f'  non-int fallback all: {ALLOW_NON_INT_FALLBACK_ALL}')
print(f'  global shortfall fill: {FILL_GLOBAL_SHORTFALL}')

# System prompt - must match CFG.system_prompt in competition notebook exactly
SYSTEM_PROMPT = (
    'You are an elite mathematical problem solver with expertise at the International '
    'Mathematical Olympiad (IMO) level. Your goal is to find the correct answer through '
    'rigorous mathematical reasoning.\n\n'
    '# Problem-Solving Approach:\n'
    '1. UNDERSTAND: Carefully read and rephrase the problem in your own words. '
    'Identify what is given, what needs to be found, and any constraints.\n'
    '2. EXPLORE: Consider multiple solution strategies. Think about relevant theorems, '
    'techniques, patterns, or analogous problems. Don\'t commit to one approach immediately.\n'
    '3. PLAN: Select the most promising approach and outline key steps before executing.\n'
    '4. EXECUTE: Work through your solution methodically. Show all reasoning steps clearly.\n'
    '5. VERIFY: Check your answer by substituting back, testing edge cases, or using '
    'alternative methods. Ensure logical consistency throughout.\n\n'
    '# Mathematical Reasoning Principles:\n'
    '- Break complex problems into smaller, manageable sub-problems\n'
    '- Look for patterns, symmetries, and special cases that provide insight\n'
    '- Use concrete examples to build intuition before generalizing\n'
    '- Consider extreme cases and boundary conditions\n'
    '- If stuck, try working backwards from the desired result\n'
    '- Be willing to restart with a different approach if needed\n\n'
    '# Verification Requirements:\n'
    '- Cross-check arithmetic and algebraic manipulations\n'
    '- Verify that your solution satisfies all problem constraints\n'
    '- Test your answer with simple cases or special values when possible\n'
    '- Ensure dimensional consistency and reasonableness of the result\n\n'
    '# Output Format:\n'
    'The final answer must be a non-negative integer between 0 and 99999.\n'
    'Place your final numerical answer inside \\boxed{}, e.g., \\boxed{42}\n\n'
    'Think step-by-step and show your complete reasoning process. Quality of reasoning '
    'is as important as the final answer.'
)
print(f'System prompt loaded ({len(SYSTEM_PROMPT)} chars)')


## 5. Helper Functions

In [ ]:
def get_accuracy(metadata, key):
    """Safely extract accuracy from nested metadata dict."""
    if not isinstance(metadata, dict):
        return 0.0
    entry = metadata.get(key, {})
    if isinstance(entry, dict):
        return float(entry.get('accuracy', 0.0))
    return 0.0


def is_integer_answer(answer):
    """Check if answer is an integer in [0, 99999].

    Stricter than v1 ? avoids float?int coercion (1.5 ? 1).
    """
    if answer is None or (isinstance(answer, float) and np.isnan(answer)):
        return False
    s = str(answer).strip().replace(',', '').replace('$', '').replace(' ', '')
    if not re.fullmatch(r'-?\d+', s):
        return False
    return ANSWER_MIN <= int(s) <= ANSWER_MAX


def detect_tir_fast(row):
    """Detect TIR from actual message content (tool_calls, tool role, python blocks).
    Falls back to metadata only if no messages available."""
    messages = row.get('messages', [])
    if not isinstance(messages, list):
        messages = []

    # Content-based detection (authoritative)
    for msg in messages:
        if not isinstance(msg, dict):
            continue
        # Explicit tool call from assistant
        if msg.get('tool_calls'):
            return True
        # Tool/ipython result message = a tool was called
        if msg.get('role') in ('tool', 'ipython'):
            return True
        # Python code block in assistant content
        content = str(msg.get('content', '') or '')
        reasoning = str(msg.get('reasoning_content', '') or '')
        if '```python' in content or '```python' in reasoning:
            return True

    # Metadata fallback ONLY if zero messages to inspect
    if not messages:
        meta = row.get('metadata', {})
        no_tool = get_accuracy(meta, 'reason_high_no_tool')
        with_tool = get_accuracy(meta, 'reason_high_with_tool')
        return with_tool > no_tool

    # Has messages but no tool evidence = pure reasoning trace
    return False

def get_difficulty(no_tool_acc, with_tool_acc):
    """Classify difficulty from accuracy metrics."""
    if no_tool_acc >= EASY_MIN:
        return 'easy'
    elif no_tool_acc >= MEDIUM_MIN:
        return 'medium'
    elif with_tool_acc >= HARD_TOOL_MIN:
        return 'hard'
    return None  # unsolvable ? skip


def dart_weight(no_tool_acc, with_tool_acc):
    """DART-Math Prop2Diff weight = fail_rate ? (1 + tir_lift)."""
    fail_rate = 1.0 - no_tool_acc
    tir_lift = max(0.0, with_tool_acc - no_tool_acc)
    return fail_rate * (1.0 + tir_lift)


def messages_are_valid(messages):
    """Require at least user->assistant structure for training rows."""
    if not isinstance(messages, list) or len(messages) < 2:
        return False
    first = messages[0] if isinstance(messages[0], dict) else {}
    second = messages[1] if isinstance(messages[1], dict) else {}
    return first.get('role') == 'user' and second.get('role') == 'assistant'


print('? Helper functions defined')


## 6. Load hard_50 Problems (Data Leakage Prevention)

In [ ]:
# Upload hard_50_math_problems_set_v6.csv
from google.colab import files

hard_50_problems = set()

try:
    uploaded = files.upload()
    fname = 'hard_50_math_problems_set_v6.csv'
    if fname in uploaded:
        hard_50_df = pd.read_csv(fname)
        # Build set of first-100-char prefixes for fuzzy matching
        for prob in hard_50_df['problem'].dropna():
            hard_50_problems.add(prob[:200].strip())
            hard_50_problems.add(prob.strip())  # also exact match
        print(f'✓ Loaded {len(hard_50_df)} hard_50 problems for leakage check')
    else:
        print('⚠ hard_50 not uploaded — skipping leakage check')
except Exception as e:
    print(f'⚠ Skipping leakage check: {e}')


def is_leaked(problem_text):
    """Check if problem overlaps with hard_50 benchmark."""
    if not hard_50_problems:
        return False
    t = str(problem_text).strip()
    return t in hard_50_problems or t[:200] in hard_50_problems

## 7. Stream and Filter Dataset

**Key fix from v1**: Uses `streaming=True` — processes rows one at a time, never loads 50GB into RAM.

**Note**: The dataset has multiple splits (`high_part00`, `high_part01`, `high_part02`, `medium`, `low`). We'll process all splits until we hit the sample cap.

In [ ]:
from collections import defaultdict
import gc
import heapq
import random
import json
import zlib
import hashlib
import os
from datetime import datetime
import numpy as np
from datasets import load_dataset
from tqdm.auto import tqdm

if 'DATASET_ID' not in globals():
    DATASET_ID = 'nvidia/Nemotron-Math-v2'

required_helpers = [
    'get_accuracy', 'messages_are_valid', 'is_leaked',
    'get_difficulty', 'detect_tir_fast', 'is_integer_answer', 'dart_weight'
]
missing_helpers = [h for h in required_helpers if h not in globals()]
if missing_helpers:
    raise RuntimeError(
        'Missing helper functions: ' + ', '.join(missing_helpers) +
        '. Run the Helper Functions cell first.'
    )

# Logging config
VERBOSE_CONSOLE = False
LOG_EVERY = 5000
LOG_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.getcwd()
LOG_FILE = os.path.join(LOG_DIR, 'pipeline_progress.log')

with open(LOG_FILE, 'w', encoding='utf-8') as lf:
    lf.write(f'[{datetime.now().isoformat()}] Pipeline logging started\n')


def log_event(message, also_console=False):
    ts = datetime.now().isoformat(timespec='seconds')
    line = f'[{ts}] {message}'
    with open(LOG_FILE, 'a', encoding='utf-8') as lf:
        lf.write(line + '\n')
    if also_console or VERBOSE_CONSOLE:
        print(line)


def compact_messages(messages):
    compact = []
    for msg in messages:
        if not isinstance(msg, dict):
            continue
        role = msg.get('role')
        # Include tool/ipython result messages - these are the Python outputs
        if role not in ('user', 'assistant', 'tool', 'ipython'):
            continue

        entry = {'role': role}

        # Prepend reasoning_content to content if present
        reasoning = str(msg.get('reasoning_content', '') or '')
        content = str(msg.get('content', '') or '')
        if reasoning and reasoning not in content:
            entry['content'] = reasoning + '\n\n' + content
        else:
            entry['content'] = content

        # Preserve tool_calls if present - this is the TIR trace
        tool_calls = msg.get('tool_calls')
        if tool_calls:
            entry['tool_calls'] = tool_calls

        compact.append(entry)
    return compact


def pack_record(record):
    payload = json.dumps(record, ensure_ascii=False, separators=(',', ':')).encode('utf-8')
    return zlib.compress(payload, level=1)


def unpack_record(blob):
    return json.loads(zlib.decompress(blob).decode('utf-8'))


print('Loading Nemotron-Math-v2 in streaming mode...')
print(f'File logging: {LOG_FILE}')
log_event('Initialized stream/filter stage', also_console=False)

all_streams = load_dataset(DATASET_ID, streaming=True)
available_splits = list(all_streams.keys())

preferred_order = ['high_part02', 'high_part01', 'high_part00', 'medium', 'low']
SPLITS = [s for s in preferred_order if s in available_splits] + [s for s in available_splits if s not in preferred_order]
log_event(f'Available splits: {available_splits}')
log_event(f'Processing order: {SPLITS}')

stats = {
    'total_seen': 0,
    'skip_unsolvable': 0,
    'skip_changed': 0,
    'skip_empty_answer': 0,
    'skip_bad_messages': 0,
    'skip_leaked': 0,
    'skip_untracked_bucket': 0,
    'kept': 0,
}

# Safety controls
STREAM_CAP_TOTAL = 650_000
CHECK_EVERY = 1000
OVERSAMPLE_FACTOR = 1.00

# Prevent first split from consuming entire budget.
SPLIT_BUDGETS = {
    'high_part02': 180_000,
    'high_part01': 180_000,
    'high_part00': 140_000,
    'medium': 100_000,
    'low': 50_000,
}
DEFAULT_SPLIT_BUDGET = 80_000

# Track integer pools as primary, non-integer as bounded fallback for all tiers.
bucket_capacity = {}
for difficulty, params in TARGET_DISTRIBUTION.items():
    total_target = params['total']
    tir_target = int(total_target * params['tir_ratio'])
    notir_target = total_target - tir_target

    bucket_capacity[(difficulty, True, True)] = max(int(tir_target * OVERSAMPLE_FACTOR), 1)
    bucket_capacity[(difficulty, False, True)] = max(int(notir_target * OVERSAMPLE_FACTOR), 1)

    fallback_factor = 0.25 if difficulty == 'hard' else 0.12
    bucket_capacity[(difficulty, True, False)] = max(int(tir_target * fallback_factor), 1)
    bucket_capacity[(difficulty, False, False)] = max(int(notir_target * fallback_factor), 1)

reservoirs = defaultdict(list)  # key -> min-heap of (score, seq_id, packed_record)
seq_id = 0


def reservoir_add(key, record, weight):
    global seq_id
    cap = bucket_capacity.get(key, 0)
    if cap <= 0:
        stats['skip_untracked_bucket'] += 1
        return

    w = max(float(weight), 1e-9)
    score = random.random() ** (1.0 / w)
    heap = reservoirs[key]
    seq_id += 1
    item = (score, seq_id, pack_record(record))

    if len(heap) < cap:
        heapq.heappush(heap, item)
        return

    if score > heap[0][0]:
        heapq.heapreplace(heap, item)


def have_enough_for_targets(res_dict):
    for difficulty, params in TARGET_DISTRIBUTION.items():
        total_target = params['total']
        tir_target = int(total_target * params['tir_ratio'])
        notir_target = total_target - tir_target

        for is_tir, target_n in [(True, tir_target), (False, notir_target)]:
            int_n = len(res_dict.get((difficulty, is_tir, True), []))
            non_n = len(res_dict.get((difficulty, is_tir, False), []))
            if difficulty == 'hard':
                if int_n + non_n < target_n:
                    return False
            else:
                if int_n < target_n:
                    return False
    return True


stop_all = False
for split_name in SPLITS:
    split_seen = 0
    split_budget = SPLIT_BUDGETS.get(split_name, DEFAULT_SPLIT_BUDGET)
    log_event(f'Start split={split_name} budget={split_budget:,}', also_console=True)

    try:
        ds = load_dataset(DATASET_ID, split=split_name, streaming=True)

        for row in tqdm(ds, desc=f'  {split_name}', leave=False, disable=not VERBOSE_CONSOLE):
            stats['total_seen'] += 1
            split_seen += 1

            if stats['total_seen'] > STREAM_CAP_TOTAL:
                log_event(f'Reached total stream cap {STREAM_CAP_TOTAL:,}; stopping', also_console=True)
                stop_all = True
                break

            if split_seen > split_budget:
                log_event(f'Reached split budget for {split_name}: {split_budget:,}')
                break

            meta = row.get('metadata', {})
            no_tool_acc = get_accuracy(meta, 'reason_high_no_tool')
            with_tool_acc = get_accuracy(meta, 'reason_high_with_tool')

            if with_tool_acc == 0.0:
                stats['skip_unsolvable'] += 1
                continue

            changed = row.get('changed_answer_to_majority', False)
            metric = meta.get('reason_high_with_tool', {}) if isinstance(meta, dict) else {}
            pass_val = metric.get('pass', 1)
            count_val = metric.get('count', 1)
            pass_rate = (pass_val / count_val) if count_val > 0 else 1.0
            if changed and pass_rate < 0.75:
                stats['skip_changed'] += 1
                continue

            answer = row.get('expected_answer', '')
            if not answer or (isinstance(answer, float) and np.isnan(answer)):
                stats['skip_empty_answer'] += 1
                continue

            raw_messages = row.get('messages', [])
            if not messages_are_valid(raw_messages):
                stats['skip_bad_messages'] += 1
                continue
            messages = compact_messages(raw_messages)
            if len(messages) < 2:
                stats['skip_bad_messages'] += 1
                continue

            if is_leaked(row.get('problem', '')):
                stats['skip_leaked'] += 1
                continue

            difficulty = get_difficulty(no_tool_acc, with_tool_acc)
            if difficulty is None:
                continue

            is_tir = detect_tir_fast(row)
            is_int_answer = is_integer_answer(answer)
            weight = dart_weight(no_tool_acc, with_tool_acc)

            prompt_text = str(messages[0].get('content', ''))
            prompt_hash = hashlib.blake2b(prompt_text.strip().encode('utf-8'), digest_size=8).hexdigest()

            record = {
                'messages': messages,
                'expected_answer': str(answer),
                'difficulty': difficulty,
                'is_tir': is_tir,
                'is_int_answer': is_int_answer,
                'weight': weight,
                'prompt_hash': prompt_hash,
                'split': split_name,
                'data_source': row.get('data_source', ''),
            }

            reservoir_add((difficulty, is_tir, is_int_answer), record, weight)
            stats['kept'] += 1

            if stats['total_seen'] % LOG_EVERY == 0:
                log_event(
                    f"progress seen={stats['total_seen']:,} kept={stats['kept']:,} "
                    f"skip_unsolvable={stats['skip_unsolvable']:,} skip_changed={stats['skip_changed']:,}"
                )

            if stats['total_seen'] % CHECK_EVERY == 0 and have_enough_for_targets(reservoirs):
                log_event('Collected enough candidates for all target buckets; early stop', also_console=True)
                stop_all = True
                break

        log_event(
            f"Finished split={split_name} split_seen={split_seen:,} total_seen={stats['total_seen']:,} kept={stats['kept']:,}",
            also_console=True,
        )

    except Exception as e:
        log_event(f'Error loading split={split_name}: {e}', also_console=True)
    finally:
        try:
            del ds
        except Exception:
            pass
        gc.collect()

    if stop_all:
        break

buckets = {}
for key, heap in reservoirs.items():
    buckets[key] = [unpack_record(blob) for _, __, blob in sorted(heap, key=lambda x: x[0], reverse=True)]

log_event(
    f"Streaming complete total_seen={stats['total_seen']:,} kept={stats['kept']:,} "
    f"skip_untracked={stats['skip_untracked_bucket']:,}",
    also_console=True,
)

print('Streaming complete')
print(f'Log file: {LOG_FILE}')


## 8. Sample According to Target Distribution

In [ ]:
def weighted_take_unique(pool, n, seen_hashes, seed=42):
    """Take up to n weighted samples, avoiding duplicate prompt_hash values."""
    if n <= 0 or len(pool) == 0:
        return []

    rng = np.random.default_rng(seed)
    taken = []
    max_iters = min(len(pool) * 2, max(1000, n * 20))
    iters = 0

    while len(taken) < n and len(pool) > 0 and iters < max_iters:
        iters += 1
        weights = np.array([x.get('weight', 1.0) for x in pool], dtype=float)
        if weights.sum() <= 0:
            weights = np.ones(len(pool), dtype=float)
        weights = weights / weights.sum()

        pick = int(rng.choice(len(pool), size=1, replace=False, p=weights)[0])
        item = pool.pop(pick)

        ph = item.get('prompt_hash')
        if ph and ph in seen_hashes:
            continue
        if ph:
            seen_hashes.add(ph)
        taken.append(item)

    return taken


print('Sampling according to target distribution (adaptive + dedup)...')
print('=' * 70)

target_total = sum(v['total'] for v in TARGET_DISTRIBUTION.values())
pool_map = {k: list(v) for k, v in buckets.items()}  # mutable copy

sampled = []
sample_report = []
# Dedup scope is per target bucket to avoid starving later buckets.
seen_hashes_by_bucket = {}
seed_base = 100
seed_step = 0

for difficulty, params in TARGET_DISTRIBUTION.items():
    total_target = params['total']
    tir_target = int(total_target * params['tir_ratio'])
    notir_target = total_target - tir_target

    print(f'{difficulty.upper()}: target={total_target:,} ({tir_target:,} TIR, {notir_target:,} no-TIR)')

    for is_tir, target_n in [(True, tir_target), (False, notir_target)]:
        int_key = (difficulty, is_tir, True)
        non_key = (difficulty, is_tir, False)
        dedup_key = (difficulty, is_tir)
        bucket_seen = seen_hashes_by_bucket.setdefault(dedup_key, set())

        int_pool = pool_map.get(int_key, [])
        non_pool = pool_map.get(non_key, [])

        int_items = weighted_take_unique(int_pool, target_n, bucket_seen, seed=seed_base + seed_step)
        seed_step += 1

        remainder = target_n - len(int_items)
        non_items = []
        allow_non_int = ALLOW_NON_INT_FALLBACK_ALL or (difficulty == 'hard')
        if remainder > 0 and allow_non_int:
            non_items = weighted_take_unique(non_pool, remainder, bucket_seen, seed=seed_base + seed_step)
            seed_step += 1

        batch = int_items + non_items
        sampled.extend(batch)

        shortfall = target_n - len(batch)
        label = 'TIR' if is_tir else 'noTIR'
        flag = 'OK' if shortfall == 0 else 'SHORT'
        print(f'  {flag:6} {label:5}: got {len(batch):,}/{target_n:,} (int={len(int_items):,}, non={len(non_items):,})')

        sample_report.append({
            'difficulty': difficulty,
            'tir': is_tir,
            'target': target_n,
            'got': len(batch),
            'int_ans': len(int_items),
            'non_int_ans': len(non_items),
            'shortfall': shortfall,
        })

remaining = target_total - len(sampled)
if remaining > 0 and FILL_GLOBAL_SHORTFALL:
    print(f'\nGlobal backfill: need {remaining:,} more unique samples...')
    remaining_pool = []
    for items in pool_map.values():
        remaining_pool.extend(items)

    # For backfill, dedup globally against what is already sampled.
    global_seen = {x.get('prompt_hash') for x in sampled if x.get('prompt_hash')}
    backfill = weighted_take_unique(remaining_pool, remaining, global_seen, seed=999)
    sampled.extend(backfill)

    sample_report.append({
        'difficulty': '_global_fill',
        'tir': None,
        'target': remaining,
        'got': len(backfill),
        'int_ans': sum(1 for x in backfill if x.get('is_int_answer', False)),
        'non_int_ans': sum(1 for x in backfill if not x.get('is_int_answer', False)),
        'shortfall': remaining - len(backfill),
    })

print(f'\nTotal sampled (unique prompts): {len(sampled):,} / {target_total:,}')


## 9. Apply Curriculum Ordering and Save

In [ ]:
# Sort easy → medium → hard (curriculum learning)
difficulty_order = {'easy': 0, 'medium': 1, 'hard': 2}
sampled.sort(key=lambda x: difficulty_order[x['difficulty']])

print(f'Writing {len(sampled):,} samples to {OUTPUT_FILE}...')

written_count = 0
skipped_boxed = 0


import re

def last_boxed_matches_answer(messages, expected_answer):
    """Return True only if the last \boxed{} in the trace matches expected."""
    all_content = ' '.join(
        str(m.get('content', '')) for m in messages
        if isinstance(m, dict) and m.get('role') == 'assistant'
    )
    boxed = re.findall(r'\\boxed\{([^}]{0,50})\}', all_content)
    if not boxed:
        return False
    last = boxed[-1].strip().replace(',', '')
    expected = str(expected_answer).strip().replace(',', '')
    return last == expected



def normalize_tool_namespace(messages):
    """Map training-time tool names to inference-time namespace."""
    normalized = []
    for msg in messages:
        if not isinstance(msg, dict):
            continue
        m = dict(msg)
        tcs = m.get('tool_calls')
        if isinstance(tcs, list):
            new_tcs = []
            for tc in tcs:
                if not isinstance(tc, dict):
                    new_tcs.append(tc)
                    continue
                tc2 = dict(tc)
                fn = tc2.get('function')
                if isinstance(fn, dict):
                    fn2 = dict(fn)
                    if fn2.get('name') == 'stateful_python_code_exec':
                        fn2['name'] = 'python'
                    tc2['function'] = fn2
                new_tcs.append(tc2)
            m['tool_calls'] = new_tcs
        normalized.append(m)
    return normalized

with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    for item in tqdm(sampled, desc='Writing'):
        # Build messages with system prompt prepended
        training_messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT}
        ] + item['messages']

        # Align tool namespace with competition inference notebook
        training_messages = normalize_tool_namespace(training_messages)

        record = {
            'messages':        training_messages,
            'problem':         item.get('problem', ''),
            'expected_answer': item['expected_answer'],
            'answer':          item['expected_answer'],
            'difficulty':      item['difficulty'],
            'is_tir':          item['is_tir'],
            'uuid':            item.get('uuid', ''),
            'data_source':     item.get('data_source', ''),
            'split':           item.get('split', ''),
        }
        f.write(json.dumps(record, ensure_ascii=False) + '\n')

print(f'✓ Saved to {OUTPUT_FILE}')

# Save metadata
metadata = {
    'total_samples':       written_count if 'written_count' in locals() else len(sampled),
    'stream_stats':        stats,
    'sample_report':       sample_report,
    'target_distribution': TARGET_DISTRIBUTION,
    'difficulty_thresholds': {
        'easy_min':       EASY_MIN,
        'medium_min':     MEDIUM_MIN,
        'hard_tool_min':  HARD_TOOL_MIN,
    },
    'curriculum_ordered':  True,
    'answer_strategy':     'adaptive_integer_primary_with_nonint_fallback_global_backfill_compressed_reservoir',
}

with open(METADATA_FILE, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'✓ Metadata saved to {METADATA_FILE}')

## 10. Validation Report

In [ ]:
print('VALIDATION REPORT')
print('='*70)

# Count by difficulty and TIR
from collections import Counter
counts = Counter((x['difficulty'], x['is_tir']) for x in sampled)

print(f'\n{"Difficulty":10} {"TIR":6} {"Got":>8} {"Target":>8} {"Status":>8}')
print('-'*50)
for difficulty, params in TARGET_DISTRIBUTION.items():
    for is_tir, label in [(True, 'TIR'), (False, 'noTIR')]:
        tgt = int(params['total'] * (params['tir_ratio'] if is_tir else 1-params['tir_ratio']))
        got = counts.get((difficulty, is_tir), 0)
        ok = '✓' if abs(got - tgt) / max(tgt,1) <= 0.05 else '⚠'
        print(f'{difficulty:10} {label:6} {got:>8,} {tgt:>8,} {ok:>8}')

print(f'\nTotal: {len(sampled):,} / {sum(p["total"] for p in TARGET_DISTRIBUTION.values()):,} target')

# Answer type breakdown
int_count = sum(1 for x in sampled if x.get('is_int_answer', is_integer_answer(x['expected_answer'])))
print(f'\nAnswer type:')
print(f'  Integer (0-99999): {int_count:,} ({int_count/len(sampled)*100:.1f}%)')
print(f'  Other format:      {len(sampled)-int_count:,} ({(len(sampled)-int_count)/len(sampled)*100:.1f}%)')

# Curriculum check
diffs = [x['difficulty'] for x in sampled]
easy_end  = next((i for i,d in enumerate(diffs) if d != 'easy'),  0)
hard_start= next((i for i,d in enumerate(reversed(diffs)) if d != 'hard'), 0)
print(f'\nCurriculum ordering:')
print(f'  First {easy_end:,} samples → easy')
print(f'  Last  {hard_start:,} samples → hard')
print(f'  ✓ Sorted easy→medium→hard')

# Sample previews
print('\nSample previews:')
print('-'*70)
for diff in ['easy', 'medium', 'hard']:
    s = next((x for x in sampled if x['difficulty'] == diff), None)
    if s:
        print(f'\n{diff.upper()}:')
        print(f'  Problem:  {str(s.get("problem", ""))[:120]}...')
        print(f'  Answer:   {s["expected_answer"]}')
        print(f'  TIR:      {s["is_tir"]}')
        print(f'  Messages: {len(s["messages"])} turns')
        print(f'  Source:   {s.get("data_source", "")}')

## 11. File Size Check and Download

In [ ]:
import os

size_mb = os.path.getsize(OUTPUT_FILE) / 1024**2
meta_mb = os.path.getsize(METADATA_FILE) / 1024**2

print(f'File sizes:')
print(f'  {OUTPUT_FILE}: {size_mb:.1f} MB')
print(f'  {METADATA_FILE}: {meta_mb:.2f} MB')

# Compress for easier download
import gzip, shutil
gz_path = OUTPUT_FILE + '.gz'
with open(OUTPUT_FILE, 'rb') as f_in:
    with gzip.open(gz_path, 'wb', compresslevel=6) as f_out:
        shutil.copyfileobj(f_in, f_out)

gz_mb = os.path.getsize(gz_path) / 1024**2
print(f'  {gz_path}: {gz_mb:.1f} MB (compressed)')
print(f'  Compression ratio: {size_mb/gz_mb:.1f}x')

print(f'\n✓ All files saved to Google Drive at {DRIVE_DIR}')
print('You can also download directly:')

from google.colab import files
files.download(METADATA_FILE)    # small, download always
# Uncomment to also download the JSONL (may be large):
# files.download(gz_path)

print('\n' + '='*70)
print('PIPELINE COMPLETE')
print('='*70)
print('Next: Upload nemotron_training_data.jsonl to Kaggle as a dataset')
print('Then run fine-tuning with Unsloth BF16 LoRA on the H100')